In [17]:
import random
import torch
from torch import nn

In [18]:
import sys
print(sys.version)
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
print('Device Name:', torch.cuda.get_device_name(0))

3.14.6 | packaged by Anaconda, Inc. | (main, Jun 18 2026, 21:28:09) [GCC 14.3.0]
PyTorch Version: 2.13.0+cu130
CUDA Available: True
Device Name: NVIDIA GeForce RTX 2060 SUPER


In [19]:
class Module(nn.Module):
    def __init__(self):
        super().__init__()

    def loss(self, y_hat, y):
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

    def training_step(self, batch):
        return self.loss(self(*batch[:-1]), batch[-1])

    def validation_step(self, batch):
        return self.loss(self(*batch[:-1]), batch[-1])

    def configure_optimizers(self):
        return SGD([self.w, self.b], self.lr)


In [20]:
class SyntheticData():
    def __init__(self, w, b, batch_size, noise = 0.01, num_of_training_data= 10000, num_of_validation_data = 1000):
        self.w = w
        self.b = b
        self.batch_size = batch_size
        self.noise = noise
        self.num_of_training_data = num_of_training_data
        self.num_of_validation_data = num_of_validation_data
        self.n = num_of_training_data + num_of_validation_data
        self.X = torch.randn(self.n, len(w))
        noise = torch.randn(self.n, 1) * self.noise
        self.Y = torch.matmul(self.X, self.w.reshape(-1,1)) + self.b + noise

    # def get_dataloader(self, train):
    #     if train:
    #         ind = list(range(0,self.num_of_training_data))
    #         random.shuffle(ind)
    #     else:
    #         ind = list(range(self.num_of_training_data, self.n))
    #     for index in range(0, len(ind), self.batch_size):
    #         yield self.X[ind[index]:ind[index + self.batch_size]], self.Y[ind[index]: ind[index + self.batch_size]]

    def get_dataloader(self, train):
        if train:
            ind = list(range(0, self.num_of_training_data))
            random.shuffle(ind)
        else:
            ind = list(range(self.num_of_training_data, self.n))

        for index in range(0, len(ind), self.batch_size):
            # 1. Grab the exact slice of indices for this batch

            batch_indices = ind[index : index + self.batch_size]
            # print(ind[index], ind[index + self.batch_size], batch_indices)
            # 2. Use advanced indexing (pass the list of indices directly to the tensor)
            # This extracts exactly the scrambled rows you want without relying on sequential slicing!
            yield self.X[batch_indices], self.Y[batch_indices]

    def train_dataloader(self):
        return self.get_dataloader(True)

    def validation_dataloader(self):
        return self.get_dataloader(False)


In [21]:
class Synthetic_Data():
    def __init__(self,w , b, batch_size, noise = 0.01, num_of_training_data = 10000, num_of_validation_data = 10000):
        self.w = w
        self.b = b
        self.batch_size = batch_size
        self.noise = noise
        self.num_training_data = num_of_training_data
        self.num_validation_data = num_of_validation_data
        self.n = num_of_training_data + num_of_validation_data
        self.X = torch.randn(self.n, len(w))
        self.noise = torch.randn(self.n, 1) * noise
        self.Y = torch.matmul(self.X, self.w.reshape(-1,1)) + self.b + self.noise

    def get_tenosorloader(self, tensors, indicies, train = True):
        tensors = tuple(a[indicies] for a in tensors)
        dataset = torch.utils.data.TensorDataset(*tensors)
        return torch.utils.data.DataLoader(dataset, batch_size=self.batch_size, shuffle=train)

    def get_dataloader(self, train):
        i = slice(0, self.num_training_data) if train else slice(self.num_training_data, self.n)
        return self.get_tenosorloader((self.X, self.Y), i , train)

    def training_dataset(self):
        return self.get_dataloader(True)

    def valodation_dataset(self):
        return self.get_dataloader(False)



In [22]:
class Trainer():
    def __init__(self, num_of_epochs):
        super().__init__()
        self.num_of_epochs = num_of_epochs

    def prepare_model(self, model):
        model.trainer = self
        self.model = model

    def prepare_data(self, data):
        self.data = data
        # self.num_train_batches = len(self.train_dataset)
        # self.num_validation_batches = len(self.validation_dataset) if self.validation_dataset is not None else 0

    def fit(self, model, data):
        self.prepare_model(model)
        self.prepare_data(data)
        self.optim = model.configure_optimizers()
        self.epoch = 0
        self.train_batch_idx = 0
        self.validation_batch_idx = 0
        for self.epoch in range(self.num_of_epochs):
            self.fit_epoch()

    def fit_epoch(self):
        for batch in self.data.train_dataloader():
            loss = self.model.training_step(batch)
            self.optim.zero_grad()
            with torch.no_grad():
                loss.backward()
                self.optim.step()
        self.train_batch_idx += 1
        if self.data.validation_dataloader() is None:
            return
        for batch in self.data.validation_dataloader():
            with torch.no_grad():
                self.model.validation_step(batch)


In [23]:
class LinearRegression(Module):
    def __init__(self,num_inputs, lr, sigma = 0.01):
        super().__init__()
        self.lr = lr
        self.sigma = sigma
        self.num_inputs = num_inputs
        self.w = torch.normal(0, sigma, (num_inputs, 1), requires_grad = True)
        self.b = torch.zeros(1, requires_grad = True)

    def forward(self, x):
        return torch.matmul(x, self.w) + self.b

    def loss(self, y_hat, y):
        return torch.mean((y_hat - y)**2 * 1/2)

In [24]:
class SGD():
    def __init__(self, params, lr):
        self.params = params
        self.lr = lr

    def step(self):
        for param in self.params:
            param -= self.lr * param.grad

    def zero_grad(self):
        for param in self.params :
            if param is not None and param.grad is not None:
                param.grad.zero_()


In [25]:
X, y = next(iter(data.get_dataloader(False)))
X[0], y[0]

(tensor([1.6382, 1.1248]), tensor([3.4613]))

In [26]:
model = LinearRegression(2, lr=0.03)
data = SyntheticData(w=torch.tensor([2,-3.4]), b = 4.2, batch_size=32)
trainer = Trainer(num_of_epochs = 10)
trainer.fit(model, data)

In [27]:
model.w, model.b

(tensor([[ 2.0002],
         [-3.4003]], requires_grad=True),
 tensor([4.1999], requires_grad=True))

In [31]:
data = Synthetic_Data(torch.tensor([2,-3.4]), torch.tensor([4]), batch_size=32)
data.training_dataset()